In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_pruning as tp

from torchvision.models import mobilenet_v2
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu126


In [2]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_dataset = ImageFolder("../data/imagenette2-160/val", transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

imagenette_wnid_to_imagenet_idx = {
    "n01440764": 0, "n02102040": 217, "n02979186": 482, "n03000684": 491,
    "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
    "n03445777": 574, "n03888257": 701,
}
imagenet_indices = [imagenette_wnid_to_imagenet_idx[wnid] for wnid in val_dataset.classes]

def evaluate_accuracy(model, dataloader, imagenet_indices, device="cpu"):
    model.eval()
    model.to(device)
    idx_tensor = torch.tensor(imagenet_indices, device=device)
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            relevant_logits = outputs[:, idx_tensor]
            predicted = torch.argmax(relevant_logits, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [3]:
train_dataset = ImageFolder("../data/imagenette2-160/train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

print("Training images:", len(train_dataset))

Training images: 9469


In [4]:
p_model = mobilenet_v2(weights="DEFAULT")
p_model.eval()

example_inputs = torch.randn(1, 3, 224, 224)
pruner = tp.pruner.MagnitudePruner(
    p_model,
    example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.2,
    ignored_layers=[p_model.classifier]
)
pruner.step()

pre_ft_accuracy = evaluate_accuracy(p_model, val_loader, imagenet_indices, "cpu")
print(f"Pre-Fine-Tuning Accuracy: {pre_ft_accuracy:.2f}%")

Pre-Fine-Tuning Accuracy: 8.38%


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
p_model.to(device)
p_model.train()

idx_tensor = torch.tensor(imagenet_indices, device=device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(p_model.parameters(), lr=0.001, momentum=0.9)

num_epochs = 3

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = p_model(images)
        relevant_logits = outputs[:, idx_tensor]
        loss = criterion(relevant_logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} — Avg Loss: {running_loss/len(train_loader):.4f}")

p_model.eval()
print("Fine-tuning complete.")

Epoch 1/3 — Avg Loss: 0.8055
Epoch 2/3 — Avg Loss: 0.4123
Epoch 3/3 — Avg Loss: 0.3061
Fine-tuning complete.


In [6]:
post_ft_accuracy = evaluate_accuracy(p_model, val_loader, imagenet_indices, "cpu")
print(f"Post-Fine-Tuning Accuracy: {post_ft_accuracy:.2f}%")

Post-Fine-Tuning Accuracy: 89.99%


In [7]:
p_model.to("cpu")
pfq_model = torch.quantization.quantize_dynamic(
    p_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

pfq_accuracy = evaluate_accuracy(pfq_model, val_loader, imagenet_indices, "cpu")
print(f"Fine-Tuned P → Q Accuracy: {pfq_accuracy:.2f}%")

C:\Users\user\AppData\Local\Temp\ipykernel_19656\323389841.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  pfq_model = torch.quantization.quantize_dynamic(


Fine-Tuned P → Q Accuracy: 90.04%
